# rPPG — FactorizePhys eğitimine Kaggle'da devam

Yerelde epoch 73'te kalan eğitimi Kaggle GPU'sunda sürdürür, sonunda 28 test videosunda değerlendirir.

**Ayarlar (sağ panel → Session options):**
- **Accelerator: GPU T4 x2** (P100 seçmeyin: güncel PyTorch P100'ü desteklemiyor olabilir)
- **Internet: On** (Hugging Face'ten veri iner; hesabın telefonla doğrulanmış olması gerekir)
- İsteğe bağlı: *Add-ons → Secrets* ile `HF_TOKEN` (Hugging Face erişim anahtarı) eklerseniz indirmeler hız sınırına takılmaz.

**Çalıştırma:** sağ üstte **Save Version → Save & Run All (Commit)**. Böylece iş Kaggle sunucusunda arka planda çalışır;
tarayıcıyı/bilgisayarı kapatabilirsiniz. Bittiğinde (≈10–11 saat) sürümün **Output** sekmesinden `cikti/` klasörünü indirin
(`best.pth`, `SONUCLAR.md`, test sonuçları, günlükler).

Adımlar: kod + rPPG-Toolbox → veri (MCD-rPPG, UBFC-rPPG, PURE; indir-işle-sil) ve test kümesi → eğitim (veri geldikçe büyür)
→ değerlendirme. Ayrıntılar: `scripts/dl_*.py`.

In [ ]:
# 1) Ortam, kod ve rPPG-Toolbox
import os, subprocess, shutil, sys, time, glob, json
from datetime import datetime, timedelta

T0 = datetime.now()
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert shutil.which("nvidia-smi"), "GPU yok: Session options -> Accelerator: GPU T4 x2 seçin"

REPO = "/kaggle/working/rppg"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Nisagwn/rppg.git", REPO], check=True)
TB = os.path.join(REPO, "external", "rPPG-Toolbox")
if not os.path.exists(TB):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ubicomplab/rPPG-Toolbox.git", TB], check=True)
os.chdir(REPO)

try:  # isteğe bağlı Hugging Face anahtarı (Add-ons -> Secrets -> HF_TOKEN)
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN kullanılıyor")
except Exception:
    print("HF_TOKEN yok (anonim indirme)")

import torch, cv2, numpy, scipy, pandas
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
# 2) Klasörler, süre ve disk bütçesi
DATA = "/tmp/dl"                      # işlenmiş veri (geçici disk; çıktıya girmez)
RAW = "/tmp/dl_raw"                   # geçici indirmeler (işlenince silinir)
OUT = "/kaggle/working/cikti"         # indirilecek sonuçlar
LOGS = os.path.join(OUT, "gunlukler")
for d in (DATA, RAW, OUT, LOGS):
    os.makedirs(d, exist_ok=True)

SESSION_H = 12                         # Kaggle oturum sınırı
TRAIN_UNTIL = T0 + timedelta(hours=SESSION_H - 1.6)   # değerlendirme + kayıt için pay
print("eğitim en geç", TRAIN_UNTIL.strftime("%H:%M"), "(sunucu saati) durur")

free = shutil.disk_usage("/tmp").free / 1e9
print(f"/tmp boş: {free:.0f} GB | /kaggle/working boş: {shutil.disk_usage('/kaggle/working').free/1e9:.0f} GB")
# MCD: ~3000 kullanılabilir video x 30 fps x 72x72x3 bayt = saniye başına ~1.4 GB
budget = free - 12 - 2 - 1 - 2         # indirme payı, test, UBFC, PURE
MAX_SEC = int(max(15, min(45, budget / 1.4)))
print("MCD video başına saklanacak süre:", MAX_SEC, "s")

# yerelde epoch 73'te kalan eğitim (en iyi: epoch 72)
if not os.path.exists(os.path.join(OUT, "last.pth")):
    shutil.copy("kaggle/baslangic_e73.pth", os.path.join(OUT, "last.pth"))
    shutil.copy("kaggle/baslangic_best_e72.pth", os.path.join(OUT, "best.pth"))

In [ ]:
# 3) Veri hazırlama arka planda: (A) MCD-rPPG 600 kişi  (B) test kümesi -> UBFC 34 denek -> PURE 60 oturum
PY = sys.executable
env = dict(os.environ, PYTHONUNBUFFERED="1")

def launch(name, args):
    log = open(os.path.join(LOGS, name + ".log"), "a")
    return subprocess.Popen([PY, "-u"] + args, cwd=REPO, stdout=log, stderr=subprocess.STDOUT, env=env)

common = ["--out", DATA, "--raw", RAW]
jobA = launch("hazirla_mcd", ["scripts/dl_prepare.py", "--mcd", "600", "--workers", "3", "--max-sec", str(MAX_SEC)] + common)
jobB = subprocess.Popen(["bash", "-c",
    f"{PY} -u scripts/dl_prepare.py --test-download --workers 2 {' '.join(common)}; "
    f"{PY} -u scripts/dl_prepare.py --ubfc 34 --workers 1 {' '.join(common)}; "
    f"{PY} -u scripts/dl_prepare.py --pure 60 --workers 1 {' '.join(common)}"],
    cwd=REPO, stdout=open(os.path.join(LOGS, "hazirla_test_ubfc_pure.log"), "a"), stderr=subprocess.STDOUT, env=env)
print("veri hazırlama başladı")

In [ ]:
# 4) Eğitim: yeterli veri gelince başlar; yeni gelen videolar her epoch başında eklenir
def n_ready():
    return sum(len(glob.glob(os.path.join(DATA, s, "*.npz"))) for s in ("mcd", "ubfc", "pure"))

while n_ready() < 300:
    if jobA.poll() is not None and jobB.poll() is not None:
        break
    print(time.strftime("%H:%M"), "hazır video:", n_ready(), flush=True)
    time.sleep(120)

train = launch("egitim", ["scripts/dl_train.py", "--data", DATA, "--out", OUT, "--epochs", "200", "--steps", "500",
                          "--deadline", TRAIN_UNTIL.strftime("%Y-%m-%d %H:%M")])
print("eğitim başladı")

In [ ]:
# 5) İzleme: 10 dakikada bir özet (Commit modunda bu hücre eğitim bitene kadar sürer)
def tail(path, n):
    try:
        return open(path, encoding="utf-8", errors="replace").read().splitlines()[-n:]
    except OSError:
        return []

while train.poll() is None:
    ep = [l for l in tail(os.path.join(LOGS, "egitim.log"), 400) if l.startswith("epoch ")]
    disk = shutil.disk_usage("/tmp").free / 1e9
    print(f"{time.strftime('%H:%M')} | veri {n_ready()} video | disk {disk:.0f} GB | "
          f"{ep[-1] if ep else 'ilk epoch sürüyor'}", flush=True)
    if disk < 4 and jobA.poll() is None:   # disk koruması
        jobA.terminate(); print("disk azaldı: MCD indirmesi durduruldu")
    time.sleep(600)
print("\n".join(tail(os.path.join(LOGS, "egitim.log"), 15)))

In [ ]:
# 6) Değerlendirme: aynı 28 test videosu (UBFC 8 + MCD 20), hazır model ve POS ile karşılaştırma
while jobB.poll() is None and len(glob.glob(os.path.join(DATA, "test", "*.npz"))) < 28:
    print("test kümesi bekleniyor:", len(glob.glob(os.path.join(DATA, "test", "*.npz"))), "/ 28", flush=True)
    time.sleep(60)
for j in (jobA, jobB):
    if j.poll() is None:
        j.terminate()

r = subprocess.run([PY, "scripts/dl_evaluate.py", "--model", os.path.join(OUT, "best.pth"),
                    "--test", os.path.join(DATA, "test"), "--data", DATA, "--out", OUT,
                    "--history", os.path.join(OUT, "gecmis.json")],
                   cwd=REPO, capture_output=True, text=True, env=env)
print(r.stdout[-6000:], r.stderr[-3000:])

In [ ]:
# 7) Çıktılar (last.pth: bir sonraki oturumda devam etmek için gerekli)
print(open(os.path.join(OUT, "SONUCLAR.md"), encoding="utf-8").read() if os.path.exists(os.path.join(OUT, "SONUCLAR.md")) else "rapor yok")
for p in sorted(glob.glob(os.path.join(OUT, "*"))):
    print(f"{os.path.getsize(p)/1e6:8.2f} MB  {p}" if os.path.isfile(p) else f"         {p}/")
shutil.rmtree(REPO, ignore_errors=True)   # çıktıda yalnızca cikti/ kalsın
print("süre:", datetime.now() - T0)